# Fitness Data — First Exploration

Goal for this notebook: load `data/fitness_data.csv` and just look at it — shape, column types, missing values — before doing any cleaning or charts.

In [4]:
import pandas as pd

df = pd.read_csv("../data/fitness_data.csv")

In [5]:
# Peek at the first few rows
df.head()

,date,workout_type,duration_min,distance_km,calories,avg_heart_rate
0,2026-03-06,Rest,0,NaN,0,NaN
1,2026-03-07,Run,34,5.67,340,174.0
2,2026-03-08,Run,43,7.18,430,162.0
3,2026-03-09,Walk,61,5.10,244,107.0
4,2026-03-10,Walk,72,6.02,288,98.0


In [3]:
# How many rows/columns, and what type is each column?
print(df.shape)
df.info()

(180, 6)
<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            180 non-null    str    
 1   workout_type    180 non-null    str    
 2   duration_min    180 non-null    int64  
 3   distance_km     86 non-null     float64
 4   calories        180 non-null    int64  
 5   avg_heart_rate  133 non-null    float64
dtypes: float64(2), int64(2), str(2)
memory usage: 8.6 KB


In [6]:
# Which columns have missing values, and how many?
df.isna().sum()

date               0
workout_type       0
duration_min       0
distance_km       94
calories           0
avg_heart_rate    47
dtype: int64

In [7]:
# Quick summary stats for the numeric columns
df.describe()

,duration_min,distance_km,calories,avg_heart_rate
count,180.000000,86.000000,180.000000,133.000000
mean,34.611111,7.625814,222.722222,128.022556
std,25.029678,5.447980,181.436079,28.141812
min,0.000000,2.100000,0.000000,80.000000
25%,0.000000,4.860000,0.000000,101.000000
50%,37.500000,5.990000,230.000000,129.000000
75%,54.250000,8.107500,348.000000,156.000000
max,89.000000,33.050000,712.000000,175.000000


## Cleaning: fix the `date` column's type

`df.info()` above shows `date` as `object`/`str` — pandas just sees it as plain text, not an actual date, since CSVs don't carry type information. We need to convert it to a real datetime so we can sort chronologically and plot trends over time later.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

# Confirm it's now a real datetime column
df.info()

## First look: how often does each workout type happen?

Why count this first? Two reasons: (1) it's a sanity check that the random data generator's weights actually landed roughly where we expected, and (2) it's the simplest possible first insight — the basic shape of the data — which matters later too (e.g. Rest days have structurally-zero duration/calories, so they shouldn't be lumped into averages the same way as real workouts).

In [ ]:
# pandas version
df["workout_type"].value_counts()

Same thing in SQL — pandas' `value_counts()` is really just a `GROUP BY` + `COUNT(*)` under the hood. We can use Python's built-in `sqlite3` to load `df` into a temporary in-memory database and query it with real SQL (no extra install needed).

In [ ]:
import sqlite3

# an in-memory database - lives only for this notebook session, not saved to disk
conn = sqlite3.connect(":memory:")
df.to_sql("workouts", conn, index=False, if_exists="replace")

# SQL version
pd.read_sql_query("""
    SELECT workout_type, COUNT(*) AS count
    FROM workouts
    GROUP BY workout_type
    ORDER BY count DESC
""", conn)

## Average duration and calories by workout type

We exclude Rest days here — their duration/calories are structurally zero (not a real "low" workout), so including them would drag down the averages for no meaningful reason.

In [ ]:
# pandas version
(
    df[df["workout_type"] != "Rest"]
    .groupby("workout_type")[["duration_min", "calories"]]
    .mean()
    .round(1)
)

SQL version — reuses the `workouts` table and `conn` connection we already set up earlier for the value_counts comparison.

In [ ]:
pd.read_sql_query("""
    SELECT
        workout_type,
        ROUND(AVG(duration_min), 1) AS avg_duration_min,
        ROUND(AVG(calories), 1) AS avg_calories
    FROM workouts
    WHERE workout_type != 'Rest'
    GROUP BY workout_type
""", conn)

## Does "fitness" improve over time?

The generator scales up distance (for a given duration) by up to ~15% from day 1 to day 180, for Run/Walk/Cycling. Comparing raw `distance_km` over time is misleading though, since duration is random each day too — a long walk on day 5 could easily beat a short walk on day 170.

Instead we use **pace** (`distance_km / duration_min`, km covered per minute), which cancels out that duration randomness and isolates just the trend. We split the 180 days into two halves and compare average pace per workout type.

In [ ]:
# pandas version
distance_types = ["Run", "Walk", "Cycling"]
midpoint = df["date"].median()

pace_df = df[df["workout_type"].isin(distance_types)].copy()
pace_df["pace_km_per_min"] = pace_df["distance_km"] / pace_df["duration_min"]
pace_df["half"] = pace_df["date"].apply(lambda d: "first_half" if d < midpoint else "second_half")

(
    pace_df
    .groupby(["workout_type", "half"])["pace_km_per_min"]
    .mean()
    .round(3)
    .unstack()
)

SQL version — splits on whether each date is before/after the midpoint of the date range, using a `CASE WHEN` inside the `GROUP BY`.

In [ ]:
# Refresh the table since `df["date"]` changed after the earlier to_sql load
df.to_sql("workouts", conn, index=False, if_exists="replace")

pd.read_sql_query("""
    SELECT
        workout_type,
        CASE
            WHEN date < (
                SELECT date(julianday(MIN(date)) + (julianday(MAX(date)) - julianday(MIN(date))) / 2)
                FROM workouts
            ) THEN 'first_half'
            ELSE 'second_half'
        END AS half,
        ROUND(AVG(distance_km / duration_min), 3) AS avg_pace_km_per_min
    FROM workouts
    WHERE workout_type IN ('Run', 'Walk', 'Cycling')
    GROUP BY workout_type, half
    ORDER BY workout_type, half
""", conn)